# 3.2 Face AutoKeras Broad Constrained Search

## Purpose

Trial 002 expands AutoKeras beyond the vanilla CNN search space while keeping compute constraints.

Compared with Trial 001:
- `block_type=None` allows AutoKeras to tune the image block type.
- Images are still streamed through `tf.data`.
- Augmentation is applied on the fly only to training batches.
- Class imbalance is handled using `class_weight`.
- Early stopping allows more epochs without wasting compute.
- The model is evaluated using recall/F1/ROC-AUC, not accuracy alone.

This experiment is accepted only if Stroke recall and Stroke F1 improve meaningfully over Trial 001.

In [1]:
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FACE_SPLIT_MANIFEST_PATH,
    FACE_IMAGE_SIZE,
    FACE_AUTOKERAS_TRIAL_002_DIR,
    FACE_AUTOKERAS_TRIAL_002_MODEL_PATH,
    FACE_AUTOKERAS_TRIAL_002_RESULTS_PATH,
    FACE_AUTOKERAS_TRIAL_002_PREDICTIONS_PATH,
    FACE_AUTOKERAS_TRIAL_002_CONFUSION_MATRIX_PATH,
    FACE_AUTOKERAS_TRIAL_002_HISTORY_PATH,
)
from rural_stroke_assist.preprocessing.tf_image_dataset import (
    dataframe_to_tf_dataset,
    add_augmentation,
)
from rural_stroke_assist.preprocessing.face_dataset import (
    compute_class_weights_from_manifest,
    INVERSE_LABEL_MAPPING,
)
from rural_stroke_assist.modeling.autokeras_models import (
    build_broad_constrained_image_automodel,
)
from rural_stroke_assist.modeling.callbacks import (
    build_early_stopping_callback,
    build_csv_logger_callback,
)
from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

c:\Users\Asus\Downloads\Uni_work\Grad-Project\autokeras_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import keras
from autokeras.preprocessors.common import AddOneDimension, CastToInt32, CastToString, LambdaPreprocessor
from autokeras.preprocessors.encoders import LabelEncoder, OneHotEncoder
from autokeras.preprocessors.postprocessors import SigmoidPostprocessor, SoftmaxPostprocessor

keras.utils.get_custom_objects().update({
    cls.__name__: cls
    for cls in [
        AddOneDimension,
        CastToInt32,
        CastToString,
        LambdaPreprocessor,
        LabelEncoder,
        OneHotEncoder,
        SigmoidPostprocessor,
        SoftmaxPostprocessor,
    ]
})

In [2]:
tf.keras.backend.clear_session()
gc.collect()

tf.random.set_seed(42)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Enabled memory growth for {gpu}")
    except Exception as exc:
        print(f"Could not set memory growth for {gpu}: {exc}")


GPUs: []


In [3]:
face_split_df = pd.read_csv(FACE_SPLIT_MANIFEST_PATH)

face_split_df.groupby(["split", "class_label"]).size()

split  class_label
test   NonStroke      208
       Stroke         110
train  NonStroke      970
       Stroke         510
val    NonStroke      208
       Stroke         109
dtype: int64

In [4]:
train_df = face_split_df[face_split_df["split"] == "train"].copy()
val_df = face_split_df[face_split_df["split"] == "val"].copy()
test_df = face_split_df[face_split_df["split"] == "test"].copy()

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 1480
Val: 317
Test: 318


In [5]:
class_weights = compute_class_weights_from_manifest(train_df)
class_weights

{0: 0.7628865979381443, 1: 1.4509803921568627}

In [6]:
IMAGE_SIZE = FACE_IMAGE_SIZE  # currently 160x160
BATCH_SIZE = 8
MAX_TRIALS = 3
EPOCHS = 20
PATIENCE = 3

IMAGE_SIZE, BATCH_SIZE, MAX_TRIALS, EPOCHS, PATIENCE

((160, 160), 8, 3, 20, 3)

In [7]:
train_ds = dataframe_to_tf_dataset(
    train_df,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)

train_ds = add_augmentation(train_ds)

val_ds = dataframe_to_tf_dataset(
    val_df,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = dataframe_to_tf_dataset(
    test_df,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [8]:
for images, labels in train_ds.take(1):
    print("Image batch:", images.shape)
    print("Label batch:", labels.shape)
    print("Pixel range:", float(tf.reduce_min(images)), float(tf.reduce_max(images)))
    print("Labels:", labels.numpy()[:10])

Image batch: (8, 160, 160, 3)
Label batch: (8,)
Pixel range: 0.0 1.047345757484436
Labels: [1 1 0 0 0 0 1 1]


In [13]:
automodel = build_broad_constrained_image_automodel(
    max_trials=MAX_TRIALS,
    project_name="face_trial_002_autokeras_broad_160",
    directory=str(FACE_AUTOKERAS_TRIAL_002_DIR),
    overwrite=False,
)

Reloading Tuner from C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\models\experiments\face\trial_002_autokeras_broad_160\face_trial_002_autokeras_broad_160\tuner0.json


In [10]:
callbacks = [
    build_early_stopping_callback(
        patience=PATIENCE,
        monitor="val_loss",
    ),
    build_csv_logger_callback(
        output_path=FACE_AUTOKERAS_TRIAL_002_HISTORY_PATH,
    ),
]

callbacks

In [14]:
history = automodel.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
)

c:\Users\Asus\Downloads\Uni_work\Grad-Project\autokeras_env\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 430 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [15]:
best_model = automodel.export_model()
best_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cast_to_float32 (CastToFloat32) │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 5, 5, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 2048)           │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification_head_1           │ (None, 1)              │             0 │
│ (Activation)                    │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 23,536,641 (89.79 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [16]:
raw_scores = best_model.predict(test_ds)
raw_scores[:5]

40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 211ms/step


array([[0.35033673],
       [0.3581015 ],
       [0.3533299 ],
       [0.36433056],
       [0.34861872]], dtype=float32)

In [17]:
if raw_scores.ndim == 1 or raw_scores.shape[-1] == 1:
    y_score = raw_scores.reshape(-1)
    y_pred = (y_score >= 0.5).astype(int)
else:
    y_score = raw_scores[:, 1]
    y_pred = raw_scores.argmax(axis=1)

y_test = test_df["class_label"].map({"NonStroke": 0, "Stroke": 1}).to_numpy()

print("y_pred counts:", np.unique(y_pred, return_counts=True))
print("y_test counts:", np.unique(y_test, return_counts=True))

y_pred counts: (array([0]), array([318], dtype=int64))
y_test counts: (array([0, 1], dtype=int64), array([208, 110], dtype=int64))


In [18]:
metrics = evaluate_binary_classifier(
    y_true=y_test,
    y_pred=y_pred,
    y_score=y_score,
)

metrics

{'accuracy': 0.6540880503144654,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.4028409090909091}

In [19]:
target_names = [INVERSE_LABEL_MAPPING[0], INVERSE_LABEL_MAPPING[1]]

report = create_classification_report_dict(
    y_true=y_test,
    y_pred=y_pred,
    target_names=target_names,
)

report

{'NonStroke': {'precision': 0.6540880503144654,
  'recall': 1.0,
  'f1-score': 0.7908745247148289,
  'support': 208.0},
 'Stroke': {'precision': 0.0,
  'recall': 0.0,
  'f1-score': 0.0,
  'support': 110.0},
 'accuracy': 0.6540880503144654,
 'macro avg': {'precision': 0.3270440251572327,
  'recall': 0.5,
  'f1-score': 0.39543726235741444,
  'support': 318.0},
 'weighted avg': {'precision': 0.4278311775641786,
  'recall': 0.6540880503144654,
  'f1-score': 0.5173015759141019,
  'support': 318.0}}

In [20]:
save_confusion_matrix_plot(
    y_true=y_test,
    y_pred=y_pred,
    class_names=target_names,
    output_path=FACE_AUTOKERAS_TRIAL_002_CONFUSION_MATRIX_PATH,
)

FACE_AUTOKERAS_TRIAL_002_CONFUSION_MATRIX_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/face_trial_002_autokeras_broad_160_confusion_matrix.png')

In [21]:
FACE_AUTOKERAS_TRIAL_002_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df = test_df.copy()
predictions_df["true_label_encoded"] = y_test
predictions_df["predicted_label_encoded"] = y_pred
predictions_df["stroke_probability"] = y_score
predictions_df["true_label"] = predictions_df["true_label_encoded"].map(INVERSE_LABEL_MAPPING)
predictions_df["predicted_label"] = predictions_df["predicted_label_encoded"].map(INVERSE_LABEL_MAPPING)
predictions_df["is_correct"] = predictions_df["true_label_encoded"] == predictions_df["predicted_label_encoded"]

predictions_df.to_csv(FACE_AUTOKERAS_TRIAL_002_PREDICTIONS_PATH, index=False)

FACE_AUTOKERAS_TRIAL_002_PREDICTIONS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/face_trial_002_autokeras_broad_160_predictions.csv')

In [22]:
save_markdown_results(
    metrics=metrics,
    report=report,
    output_path=FACE_AUTOKERAS_TRIAL_002_RESULTS_PATH,
    title="Face Trial 002 AutoKeras Broad Constrained Search Results",
)

FACE_AUTOKERAS_TRIAL_002_RESULTS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/experiments/face_trial_002_autokeras_broad_160_results.md')

In [23]:
stroke_recall = report["Stroke"]["recall"]
stroke_f1 = report["Stroke"]["f1-score"]
roc_auc = metrics.get("roc_auc", None)

accepted = (
    stroke_recall >= 0.50
    and stroke_f1 >= 0.40
    and roc_auc is not None
    and roc_auc >= 0.65
)

accepted

False

In [24]:
if accepted:
    FACE_AUTOKERAS_TRIAL_002_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    best_model.save(FACE_AUTOKERAS_TRIAL_002_MODEL_PATH)
    print("Accepted candidate model saved to:", FACE_AUTOKERAS_TRIAL_002_MODEL_PATH)
else:
    print("Model rejected as candidate baseline. Results and predictions are saved, but model artifact is not saved.")

Model rejected as candidate baseline. Results and predictions are saved, but model artifact is not saved.
